### Dependency Compatibility Issue

The official DreamerV3 implementation requires specific library versions
(e.g., `ale_py==0.9.0` and `JAX==0.4.33`) that are not fully compatible
with Google Colab's default Python 3.13 environment.

Additionally, the `dm_control` installation fails because its `labmaze`
dependency encounters compilation issues with Python 3.13.

**Solution:** Use an older Google Colab runtime (`2026.07`) with Python 3.12
to ensure better compatibility with DreamerV3's dependencies.

In [ ]:
import sys
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define your folder
folder = '/content/drive/MyDrive/INF8250AE_research_project/'

# Add the folder to Python's search path
if folder not in sys.path:
    sys.path.insert(0, folder)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#@title Import dreamerv3 and deep min control suite
!git clone https://github.com/danijar/dreamerv3.git
%cd /content/dreamerv3

# Deep mind control suite
!pip install -U -r requirements.txt
!pip install dm_control

fatal: destination path 'dreamerv3' already exists and is not an empty directory.
/content/dreamerv3
  Using cached chex-0.1.92-py3-none-any.whl.metadata (19 kB)
  Using cached optax-0.2.9-py3-none-any.whl.metadata (7.9 kB)
  Using cached absl_py-2.5.1-py3-none-any.whl.metadata (3.3 kB)
INFO: pip is looking at multiple versions of chex to determine which version is compatible with other requirements. This could take a while.
  Using cached chex-0.1.91-py3-none-any.whl.metadata (18 kB)
INFO: pip is looking at multiple versions of optax to determine which version is compatible with other requirements. This could take a while.
  Using cached optax-0.2.8-py3-none-any.whl.metadata (7.9 kB)
  Using cached optax-0.2.7-py3-none-any.whl.metadata (7.9 kB)
  Using cached optax-0.2.6-py3-none-any.whl.metadata (7.6 kB)
  Using cached dm_control-1.0.52-py3-none-any.whl.metadata (1.5 kB)
  Using cached dm_env-1.6-py3-none-any.whl.metadata (966 bytes)
  Using cached labmaze-1.0.6-cp312-cp312-manylinux

In [ ]:
#@title Cartpole environment

from dm_control import suite

# Load the environment
env = suite.load("cartpole", "swingup")

# Initialize an episode
timestep = env.reset()

# Display observations and action space
print("Observations:")
print(timestep.observation)

print("\nAction space:")
print(env.action_spec())

/usr/local/lib/python3.12/dist-packages/glfw/__init__.py:917: GLFWError: (65550) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)


Observations:
OrderedDict({'position': array([ 0.00170804, -0.99999478,  0.00323172]), 'velocity': array([-0.01433395, -0.00429183])})

Action space:
BoundedArray(shape=(1,), dtype=dtype('float64'), name=None, minimum=[-1.], maximum=[1.])


In [ ]:
#@title Jax
import jax

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())

JAX version: 0.4.33
Devices: [CudaDevice(id=0)]


In [ ]:
!nvidia-smi

Wed Nov 18 22:08:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             28W /   70W |     107MiB /  15360MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Graphic interface

!apt-get update -qq
!apt-get install -y libosmesa6 libgl1-mesa-glx

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
The following additional packages will be installed:
  libegl-mesa0 libgbm1 libgl1-mesa-dri libglapi-mesa libglx-mesa0
Recommended packages:
  libgl1-amber-dri
The following NEW packages will be installed:
  libosmesa6
The following packages will be upgraded:
  libegl-mesa0 libgbm1 libgl1-mesa-dri libglapi-mesa libglx-mesa0
5 upgraded, 1 newly installed, 0 to remove and 265 not upgraded.
Need to get 12.3 MB of archives.
After this operation, 12.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libegl-mesa0 amd64 23.2.1-1ubuntu3.1~22.04.4 [118 kB]
Get:2 http://archive.ubuntu.co

In [1]:
# DreamerV3 debugging experiment

LOG_DIR = "/content/drive/MyDrive/INF8250AE_research_project/logs/debug"

!MUJOCO_GL=osmesa python dreamerv3/main.py \
    --configs dmc_proprio debug \
    --task dmc_cartpole_swingup \
    --env.dmc.image False \
    --run.steps 1000 \
    --logdir {LOG_DIR}

python3: can't open file '/content/dreamerv3/main.py': [Errno 2] No such file or directory


In [ ]:
# Available metrics

from pathlib import Path
import pandas as pd

LOG_DIR = Path(LOG_DIR)

metrics = pd.read_json(
    LOG_DIR / "metrics.jsonl",
    lines=True
)

print(metrics.columns.tolist())
metrics.tail()

['step', 'train/adv', 'train/adv_mag', 'train/adv_std', 'train/con', 'train/dyn_ent', 'train/ent/action', 'train/loss/con', 'train/loss/dyn', 'train/loss/policy', 'train/loss/position', 'train/loss/rep', 'train/loss/repval', 'train/loss/rew', 'train/loss/value', 'train/loss/velocity', 'train/opt/grad_norm', 'train/opt/grad_rms', 'train/opt/loss', 'train/opt/param_count', 'train/opt/param_rms', 'train/opt/update_rms', 'train/opt/updates', 'train/rep_ent', 'train/ret', 'train/ret_max', 'train/ret_min', 'train/ret_rate', 'train/rew', 'train/slowval', 'train/tar', 'train/val', 'train/weight', 'replay/items', 'replay/chunks', 'replay/streams', 'replay/ram_gb', 'replay/inserts', 'replay/samples', 'replay/updates', 'replay/replay_ratio', 'usage/psutil/proc_cpu_usage', 'usage/psutil/proc_ram_frac', 'usage/psutil/proc_ram_gb', 'usage/psutil/total_cpu_count', 'usage/psutil/total_cpu_frac', 'usage/psutil/total_ram_frac', 'usage/psutil/total_ram_total_gb', 'usage/psutil/total_ram_used_gb', 'usage/

,step,train/adv,train/adv_mag,train/adv_std,train/con,train/dyn_ent,train/ent/action,train/loss/con,train/loss/dyn,train/loss/policy,...,usage/psutil/total_ram_used_gb,usage/psutil/total_ram_avail_gb,usage/nvsmi/compute_min/gpu0,usage/nvsmi/compute_avg/gpu0,usage/nvsmi/compute_max/gpu0,usage/nvsmi/memory_min/gpu0,usage/nvsmi/memory_avg/gpu0,usage/nvsmi/memory_max/gpu0,fps/policy,fps/train
3,492,0,0,0,0.419664,2.747546,1.303943,0.833827,1,-0.000020,...,3.922718,8.422695,0,0,0,0,0,0,19.819875,161.495335
4,600,0,0,0,0.420714,2.748188,1.304006,0.834356,1,-0.000020,...,3.922161,8.423256,0,0,0,0,0,0,20.305335,165.451396
5,720,0,0,0,0.419931,2.748203,1.304070,0.833705,1,-0.000020,...,3.913002,8.432415,0,0,0,0,0,0,23.444893,187.559150
6,816,0,0,0,0.419583,2.749784,1.304130,0.851137,1,-0.000019,...,3.920197,8.425217,0,0,0,0,0,0,18.808397,141.062810
7,936,0,0,0,0.420379,2.750680,1.304265,0.839768,1,-0.000020,...,3.930382,8.415035,0,0,0,0,0,0,22.637730,181.102089
